# Welcome to NER Finetuning with HF Transformers 

## Contents 
1. Loading/Preparing Model and Dataset
2. Train (Finetune) a NER Model
3. Evaluation
4. Optimization [to be checked or deleted]

In [1]:
# import modules
import config
import train
import eval_opt
import merge_datasets

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train.set_torch_device()

device(type='cuda')

## 1. Loading/Preparing Model and Dataset

In this section, we set all of the configurations for the following steps (training, evaluation, optimization). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

### Load Model

You can choose e.g. one of the following models and add them as `path` in the model config:
* `distilbert/distilbert-base-uncased`
* `FacebookAI/xlm-roberta-base`
* `FacebookAI/roberta-base`
* `dbmdz/bert-base-german-cased`
* `dbmdz/bert-tiny-historic-multilingual-cased`
* `dbmdz/bert-mini-historic-multilingual-cased`
* `dbmdz/bert-base-historic-multilingual-cased`
* `dbmdz/electra-base-german-europeana-cased-discriminator`
* `deepset/gbert-base`

For testing purposes it is best to use a smaller model such as `dbmdz/bert-tiny-historic-multilingual-cased`

In [3]:
model = config.Resource(path="dbmdz/bert-base-german-europeana-uncased", source="hf")
model.set_name()
print(model.info())

bert-base-german-europeana-uncased will be loaded from dbmdz/bert-base-german-europeana-uncased (via hf).


### Load, merge and update dataset(s)

You can choose e.g. one of the following dataset configurations:
* zefys2025: `path="data/zefys2025_with_last_sents.hf", source="local"`
* conll2003: `path="eriktks/conll2003", source="hf"`
* germeval2014: `path="GermanEval/germeval_14", source="hf"`
* hisgermaner: `path="data/hisgermaner_not-casted.hf", source="local"`
* hipe2020: `path="data/hipe2020_not-casted.hf", source="local"`
* neiss: `path="data/neiss_arendt.hf", source="local"` and `path="data/neiss_sturm.hf", source="local"`
* europeana: `path="data/europeana_lft.hf", source="local"` and `path="data/europeana_onb.hf", source="local"`
* newseye (deprecated): `path="data/newseye.hf", source="local"`


In [15]:
neiss_sturm_ds = config.Resource(path="../data/neiss_sturm_ds.hf", source="local")
neiss_sturm_ds.set_name()
print(neiss_sturm_ds.info())
neiss_sturm_dataset = train.load_ner_dataset(neiss_sturm_ds.path, neiss_sturm_ds.source)
neiss_sturm_dataset["train"][1]

zefys2025_with_last_sents will be loaded from ../data/zefys2025_with_last_sents.hf (via local).


In [17]:
#merge datasets (input is a list of 1 to n datasets)
neiss_merged = merge_datasets.merge_ds([neiss_sturm_dataset])

In [18]:
#update ner labels and map them to zefys label list
neiss_mc_updated = merge_datasets.map_ner_tags_to_zefys(neiss_merged)

Casting the dataset: 100%|████████████████████████████████████████████████| 1663/1663 [00:00<00:00, 21974.79 examples/s]


In [26]:
#check dataset transformation correctness
zefys_label_list = ["B-LOC", "I-LOC", "B-PER", "I-PER", "B-ORG", "I-ORG", "O"]
for token, tag in zip(neiss_mc_updated["train"][24]["tokens"], neiss_mc_updated["train"][24]["ner_tags"]):
    print(token, zefys_label_list[tag])

Ueber O
das O
Geschäft O
im O
laufen O
den O
Jahre O
berichtete O
der O
Vorsitzende O
, O
Bankier O
Zuckermandel B-PER
, O
auf O
eine O
Anfrage O
, O
die O
für O
das O
Jahr O
1907 O
angekauiten O
Rohteermengen O
seien O
etwa O
um O
20 O
% O
grösser O
als O
die O
im O
Jahre O
1906 O
ver O
arbeiteten O
, O
die O
Einkaufspreise O
werden O
sich O
im O
Durchschnitt O
etwas O
niedriger O
steilen O
. O


In [27]:
#tokenize dataset and get final label list
#for roberta, tokenizer must be called with add_prefix_space=True

tokenizer = train.get_tokenizer(model.path)
tokenized_dataset = train.prepare_dataset(neiss_mc_updated, tokenizer)
label_list = train.get_label_list(neiss_mc_updated)

Map: 100%|████████████████████████████████████████████████████████████████| 1663/1663 [00:00<00:00, 13037.57 examples/s]


In [28]:
if label_list != zefys_label_list:
    print("the label_list is not identical to zefys_label_list")
else:
    print("label_list and zefys_label_list are identical")

label_list and zefys_label_list are identical


## 2. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

### Training Parameters

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [29]:
training_params = config.TrainingParams()
training_params.batch_size = 32
training_params.num_train_epochs = 1
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 1,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [32]:
trained_ner_model, model_out_path, best_result = train.train_model(model, neiss_sturm_ds, label_list, training_params, tokenized_dataset, tokenizer, save_strategy="epoch", exp_model_path="model_output")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dbmdz/bert-base-german-europeana-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,F1 Early,Model,Train,Train Params
1,No log,0.112861,0.652746,0.704998,0.677867,0.964515,0.680000,bert-base-german-europeana-uncased,"Resource(path='../data/zefys2025_with_last_sents.hf', source='local', name='zefys2025_with_last_sents')","TrainingParams(batch_size=32, eval_strategy='epoch', learning_rate=2e-05, num_train_epochs=1, weight_decay=0.01, warmup_steps=100)"


## 3. Evaluation

In [33]:
class_report, errors = eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 

              precision    recall  f1-score   support

         LOC       0.72      0.84      0.78       948
         ORG       0.37      0.37      0.37       517
         PER       0.69      0.75      0.72       804

   micro avg       0.64      0.70      0.67      2269
   macro avg       0.59      0.65      0.62      2269
weighted avg       0.63      0.70      0.66      2269

